In [1]:
import os
import pandas as pd
import random
import numpy as np
from scipy.stats import expon
from lightning.pytorch import seed_everything

## Data Augmentation
### Slicing
For each curve which is indexed by `sample_target_idx`, slice the curve into intervals of 25 cycles each starting from the 5th cycle.

In [ ]:
def generate_slices(df, col_name):
    sliced_df = pd.DataFrame({'sample_target_idx': np.repeat(df.sample_target_idx.unique(),25),
                              'label': np.repeat(df.label.unique(), 25),
                              'data_type': np.repeat(df.data_type.unique(), 25),
                              'split': np.repeat(df.nonempty_data_type_1.unique(), 25),
                              'cycle_no': range(1,26)
                             })
    for i in range(4):
        sliced_df[f'slice{i}'] = df[col_name][(i*5): (25 + i*5)].values
    return sliced_df


### Shiftinng

`data` is curve_df. For each cycle, we estimate the distribution of Rn value at that cycle using exponential family distributions. Then, for each curve which is indexed by `sample_target_idx`, we randomly pick a `k` value between 0-39 and shift the curve forward by k cycles. After the shifting, we impute back the cycles at the beginning of the curves using our estimated distributions of those first `k` cycles.


In [ ]:
def shift_k_cycles(df, loc, scale):
    '''
    Shift the curve forward by k cycles and impute back the first k cycles using estimated distribution
    '''
    k = df.k.max()
    series = df.normalized
    truncated_series = series[:(40 - k)]
    added_series = np.zeros(k)
    for i in range(k):
        added_series[i] = expon.rvs(loc[i], scale[i])
    df['shifted_normalized'] = np.concatenate([added_series, truncated_series])
    return df

# estimate distribution of each cycle using exponential family
loc = np.zeros(40)
scale = np.zeros(40)
for i in range(40):
    loc[i], scale[i] = expon.fit(data[data.cycle_no == (i+1)].normalized)

# for each curve, generate k randomly
seed_everything(2021)
k_df = pd.DataFrame({'sample_target_idx': data.sample_target_idx.unique()})
ks = random.choices(list(range(40)), k = k_df.shape[0])
k_df.loc[:,'k'] = ks

train_data = (data
                .merge(k_df, how = 'inner', on = 'sample_target_idx'))

train_data = (train_data
                .groupby('sample_target_idx')
                .apply(shift_k_cycles)
                .reset_index(drop=True))


## GRU
GRU input is a matrix of dimension (30,10) generated based on the following photo

![GRU input](./data/GRU_inputs.png')

In [ ]:
def normalize(df):
    df['normalized'] = (df
                        .groupby('sample_target_idx', group_keys = False)
                        .apply(lambda x: 100*(x.rn - x.rn.min())/
                                            (x.rn.max() - x.rn.min())))
    return df

def generate_time_lags(df, col_name, n_lags):
    df_n = df.copy()
    for n in range(1, n_lags + 1):
        df_n[f"lag{n}"] = df_n[col_name].shift(n)
    df_n = df_n.iloc[n_lags:]
    return df_n

def generate_lag(df, data_type_col, data_col):
    if 'sample_target_slice_idx' in df.columns:
        diff_df = df[['sample_target_slice_idx','sample_target_idx','cycle_no',data_type_col,'label',data_col]].copy()
        for i in range(10):
            diff_df[f'lag{i+1}'] = (df.iloc[:,(4+i+1)] - df.iloc[:,(4+i)])
    else:
        diff_df = df[['sample_target_idx','cycle_no',data_type_col,'label',data_col]].copy()
        for i in range(10):
            diff_df[f'lag{i+1}'] = (df.iloc[:,(4+i+1)] - df.iloc[:,(4+i)])
    return diff_df

def prepare_data(df,data_type_col,data_col):
    # normalize Rn 
    if not 'normalized' in df.columns:
        df = normalize(df)
    # convert label column to be between 0 and 1
    df['label'] = 1.0*(df['label'] > 0)
    if 'sample_target_slice_idx' in df.columns:
        # turn data to wide format
        wide_df = (df[['sample_target_slice_idx', 'sample_target_idx','cycle_no',data_type_col,'label',data_col]]
                        .groupby('sample_target_slice_idx')
                        .apply(generate_time_lags, col_name = data_col, n_lags = 10)
                        .reset_index(drop = True))
    else:
        # turn data to wide format
        wide_df = (df[['sample_target_idx','cycle_no',data_type_col,'label',data_col]]
                        .groupby('sample_target_idx')
                        .apply(generate_time_lags, col_name = data_col, n_lags = 10)
                        .reset_index(drop = True))
                        
    # create data feature columns                     
    diff_df = generate_lag(wide_df, data_type_col, data_col)
    # only take the last 30 cycles in case some curves have 40 or 45 cycles
    long_pcr_ids = diff_df[diff_df.cycle_no == 45].sample_target_idx.unique()
    short_diff_df = diff_df[~(diff_df.sample_target_idx.isin(long_pcr_ids)) | (diff_df.cycle_no >= 16)]
 
    return short_diff_df

### VAE
VAE input is just the last 40 cycles of normalized Rn.